In [ ]:
# Character Segmentation for Script-to-Text

# This notebook provides comprehensive character segmentation techniques for extracting individual characters from handwritten text images using OpenCV.

# ## Overview
# - Image preprocessing and binarization
# - Connected component analysis
# - Character extraction and bounding boxes
# - Contour-based segmentation
# - Character classification preparation

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
import os
from scipy import ndimage
from collections import defaultdict

# Set the image path
image_path = '/Users/aaronbaggot/Desktop/Script2Text/data/samples/Handwritten_2025-09-18_110153.png'

# Load image
img = cv2.imread(image_path)
if img is None:
    raise FileNotFoundError(f"Image not found at {image_path}")

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Display original
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(gray, cmap='gray')
plt.title('Grayscale')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Image shape: {img.shape}")
print(f"Image loaded successfully")

In [ ]:
# Step 1: Preprocessing and Binarization

# Denoise
denoised = cv2.fastNlMeansDenoising(gray, None, h=10, templateWindowSize=7, searchWindowSize=21)

# Apply Gaussian Blur
blurred = cv2.GaussianBlur(denoised, (5, 5), 0)

# Adaptive thresholding for better binarization
binary = cv2.adaptiveThreshold(
    blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV, 31, 4
)

# Morphological operations to clean up
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
cleaned = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=1)
cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, kernel, iterations=1)

# Display preprocessing steps
plt.figure(figsize=(15, 4))

plt.subplot(1, 4, 1)
plt.imshow(denoised, cmap='gray')
plt.title('Denoised')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(blurred, cmap='gray')
plt.title('Blurred')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(binary, cmap='gray')
plt.title('Binary (Adaptive)')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(cleaned, cmap='gray')
plt.title('Cleaned')
plt.axis('off')

plt.tight_layout()
plt.show()

print("Preprocessing completed")

In [ ]:
# Step 2: Connected Components Analysis

# Find connected components
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
    cleaned, connectivity=8
)

print(f"Number of components found: {num_labels}")
print(f"Stats shape: {stats.shape}")
print(f"Centroids shape: {centroids.shape}")

# Filter components by size (remove noise and very large components)
min_size = 20  # Minimum pixel area for a character
max_size = 5000  # Maximum pixel area for a character

# Component statistics: x, y, width, height, area
valid_components = []
for i in range(1, num_labels):  # Skip background (0)
    area = stats[i, cv2.CC_STAT_AREA]
    if min_size < area < max_size:
        valid_components.append(i)

print(f"Valid components after filtering: {len(valid_components)}")

# Create mask with only valid components
filtered_mask = np.zeros_like(labels)
for comp_id in valid_components:
    filtered_mask[labels == comp_id] = 255

# Visualize
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(labels, cmap='nipy_spectral')
plt.title(f'All Components ({num_labels - 1} total)')
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(filtered_mask, cmap='gray')
plt.title(f'Filtered Components ({len(valid_components)} valid)')

plt.tight_layout()
plt.show()

In [ ]:
# Step 3: Extract Character Bounding Boxes

characters = []
bounding_boxes = []

for comp_id in valid_components:
    x = stats[comp_id, cv2.CC_STAT_LEFT]
    y = stats[comp_id, cv2.CC_STAT_TOP]
    w = stats[comp_id, cv2.CC_STAT_WIDTH]
    h = stats[comp_id, cv2.CC_STAT_HEIGHT]
    area = stats[comp_id, cv2.CC_STAT_AREA]
    
    # Extract character region
    char_region = cleaned[y:y+h, x:x+w]
    
    characters.append({
        'image': char_region,
        'x': x,
        'y': y,
        'width': w,
        'height': h,
        'area': area,
        'centroid': centroids[comp_id]
    })
    
    bounding_boxes.append((x, y, w, h))

# Sort by x-coordinate (left to right reading order)
characters_sorted = sorted(characters, key=lambda c: c['x'])
bounding_boxes_sorted = sorted(bounding_boxes, key=lambda b: b[0])

print(f"Extracted {len(characters_sorted)} characters")
print(f"Bounding box format: (x, y, width, height)")

# Display first 10 bounding boxes
for i, char in enumerate(characters_sorted[:10]):
    print(f"Char {i}: x={char['x']}, y={char['y']}, w={char['width']}, h={char['height']}, area={char['area']}")

In [ ]:
# Visualize extracted characters as a grid

n_chars = min(20, len(characters_sorted))
cols = 10
rows = (n_chars + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 2))
axes = axes.flatten() if n_chars > 1 else [axes]

for idx, char in enumerate(characters_sorted[:n_chars]):
    char_img = char['image']
    axes[idx].imshow(char_img, cmap='gray')
    axes[idx].set_title(f"Char {idx}", fontsize=8)
    axes[idx].axis('off')

# Hide unused subplots
for idx in range(n_chars, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print(f"Displayed first {n_chars} characters")

In [ ]:
# Visualize bounding boxes on original image

img_with_boxes = img.copy()

for idx, (x, y, w, h) in enumerate(bounding_boxes_sorted):
    # Draw rectangle
    cv2.rectangle(img_with_boxes, (x, y), (x + w, y + h), (0, 255, 0), 2)
    # Add index label
    cv2.putText(img_with_boxes, str(idx), (x, y - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(img_with_boxes, cv2.COLOR_BGR2RGB))
plt.title(f'Character Segmentation: {len(characters_sorted)} Characters Detected')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Visualization complete with {len(characters_sorted)} bounding boxes")

In [ ]:
# Alternative: Contour-based Segmentation

contours, hierarchy = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Filter contours by area
min_contour_area = 20
max_contour_area = 5000

valid_contours = []
for contour in contours:
    area = cv2.contourArea(contour)
    if min_contour_area < area < max_contour_area:
        valid_contours.append(contour)

print(f"Found {len(contours)} total contours")
print(f"Valid contours: {len(valid_contours)}")

# Sort contours by x-coordinate
contour_data = []
for contour in valid_contours:
    x, y, w, h = cv2.boundingRect(contour)
    contour_data.append({
        'contour': contour,
        'x': x,
        'y': y,
        'w': w,
        'h': h
    })

contour_data_sorted = sorted(contour_data, key=lambda c: c['x'])

# Draw contours on image
img_contours = img.copy()
for idx, data in enumerate(contour_data_sorted):
    cv2.drawContours(img_contours, [data['contour']], 0, (0, 255, 0), 2)
    x, y, w, h = data['x'], data['y'], data['w'], data['h']
    cv2.putText(img_contours, str(idx), (x, y - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(img_contours, cv2.COLOR_BGR2RGB))
plt.title(f'Contour-Based Segmentation: {len(valid_contours)} Characters')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Character Normalization and Feature Extraction

target_size = (32, 32)  # Standard size for character recognition

normalized_chars = []

for idx, char in enumerate(characters_sorted):
    char_img = char['image']
    
    # Pad to make square
    h, w = char_img.shape
    max_dim = max(h, w)
    
    # Create square canvas
    square_img = np.zeros((max_dim, max_dim), dtype=np.uint8)
    y_offset = (max_dim - h) // 2
    x_offset = (max_dim - w) // 2
    square_img[y_offset:y_offset+h, x_offset:x_offset+w] = char_img
    
    # Resize to target size
    resized = cv2.resize(square_img, target_size, interpolation=cv2.INTER_LINEAR)
    
    # Extract features
    # 1. Fill percentage
    fill_percent = np.sum(resized > 0) / (target_size[0] * target_size[1])
    
    # 2. Aspect ratio
    h_char, w_char = char['height'], char['width']
    aspect_ratio = w_char / (h_char + 1e-6)
    
    # 3. Compactness (perimeter / area)
    contour = cv2.findContours(char_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
    if len(contour) > 0:
        perimeter = cv2.arcLength(contour[0], True)
        compactness = perimeter / (char['area'] + 1e-6)
    else:
        compactness = 0
    
    normalized_chars.append({
        'normalized': resized,
        'fill_percent': fill_percent,
        'aspect_ratio': aspect_ratio,
        'compactness': compactness,
        'original_area': char['area']
    })

print(f"Normalized {len(normalized_chars)} characters to {target_size}")

# Display normalized characters
n_show = min(15, len(normalized_chars))
cols = 5
rows = (n_show + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(12, rows * 2))
axes = axes.flatten()

for idx, norm_char in enumerate(normalized_chars[:n_show]):
    axes[idx].imshow(norm_char['normalized'], cmap='gray')
    axes[idx].set_title(f"A:{norm_char['aspect_ratio']:.2f}", fontsize=8)
    axes[idx].axis('off')

for idx in range(n_show, len(axes)):
    axes[idx].axis('off')

plt.suptitle(f'Normalized Characters ({target_size[0]}x{target_size[1]})')
plt.tight_layout()
plt.show()

# Print feature statistics
print("\nFeature Statistics:")
print(f"Fill %: min={min(c['fill_percent'] for c in normalized_chars):.3f}, "
      f"max={max(c['fill_percent'] for c in normalized_chars):.3f}")
print(f"Aspect Ratio: min={min(c['aspect_ratio'] for c in normalized_chars):.3f}, "
      f"max={max(c['aspect_ratio'] for c in normalized_chars):.3f}")
print(f"Compactness: min={min(c['compactness'] for c in normalized_chars):.3f}, "
      f"max={max(c['compactness'] for c in normalized_chars):.3f}")

In [ ]:
# Save extracted characters to disk

output_dir = '/Users/aaronbaggot/Desktop/Script2Text/output/characters'
os.makedirs(output_dir, exist_ok=True)

for idx, norm_char in enumerate(normalized_chars):
    filename = os.path.join(output_dir, f'char_{idx:04d}.png')
    cv2.imwrite(filename, norm_char['normalized'])

print(f"Saved {len(normalized_chars)} characters to {output_dir}")

# Also save a reference image with bounding boxes
reference_path = os.path.join(output_dir, 'reference_with_boxes.png')
cv2.imwrite(reference_path, img_with_boxes)
print(f"Saved reference image to {reference_path}")

In [ ]:
# Reusable Character Segmentation Function

def segment_characters(image_path, min_size=20, max_size=5000, target_size=(32, 32)):
    """
    Segment characters from a handwritten text image.
    
    Args:
        image_path: Path to input image
        min_size: Minimum component area (pixels)
        max_size: Maximum component area (pixels)
        target_size: Output character size (height, width)
    
    Returns:
        dict: Contains segmented characters, bounding boxes, and metadata
    """
    # Load and preprocess
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, None, h=10, templateWindowSize=7, searchWindowSize=21)
    blurred = cv2.GaussianBlur(denoised, (5, 5), 0)
    
    # Binarization
    binary = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 31, 4
    )
    
    # Morphology
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    cleaned = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=1)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, kernel, iterations=1)
    
    # Connected components
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(cleaned, connectivity=8)
    
    # Filter components
    valid_components = []
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if min_size < area < max_size:
            valid_components.append(i)
    
    # Extract characters
    characters = []
    for comp_id in valid_components:
        x = stats[comp_id, cv2.CC_STAT_LEFT]
        y = stats[comp_id, cv2.CC_STAT_TOP]
        w = stats[comp_id, cv2.CC_STAT_WIDTH]
        h = stats[comp_id, cv2.CC_STAT_HEIGHT]
        char_region = cleaned[y:y+h, x:x+w]
        
        characters.append({
            'image': char_region,
            'bbox': (x, y, w, h),
            'area': stats[comp_id, cv2.CC_STAT_AREA]
        })
    
    # Sort by x-coordinate
    characters_sorted = sorted(characters, key=lambda c: c['bbox'][0])
    
    # Normalize
    normalized = []
    for char in characters_sorted:
        char_img = char['image']
        h_char, w_char = char_img.shape
        max_dim = max(h_char, w_char)
        
        square_img = np.zeros((max_dim, max_dim), dtype=np.uint8)
        y_off = (max_dim - h_char) // 2
        x_off = (max_dim - w_char) // 2
        square_img[y_off:y_off+h_char, x_off:x_off+w_char] = char_img
        
        resized = cv2.resize(square_img, target_size, interpolation=cv2.INTER_LINEAR)
        normalized.append(resized)
    
    return {
        'original': img,
        'characters': characters_sorted,
        'normalized': normalized,
        'count': len(characters_sorted),
        'preprocessed': cleaned
    }

# Test the function
result = segment_characters(image_path)
print(f"Segmentation complete: {result['count']} characters found")

In [ ]:
# Summary and Analysis

print("="*60)
print("CHARACTER SEGMENTATION SUMMARY")
print("="*60)
print(f"\nImage: {image_path.split('/')[-1]}")
print(f"Original size: {img.shape}")
print(f"Total characters detected: {result['count']}")
print(f"\nBounding Box Statistics:")

bboxes = [char['bbox'] for char in result['characters']]
widths = [bbox[2] for bbox in bboxes]
heights = [bbox[3] for bbox in bboxes]

print(f"  Width - Min: {min(widths)}, Max: {max(widths)}, Avg: {np.mean(widths):.1f}")
print(f"  Height - Min: {min(heights)}, Max: {max(heights)}, Avg: {np.mean(heights):.1f}")

print(f"\nNormalized size: 32x32 pixels")
print(f"Output directory: {output_dir}")
print(f"Saved characters: {len(normalized_chars)}")

print("\n" + "="*60)
print("SEGMENTATION TECHNIQUES USED:")
print("="*60)
print("1. Adaptive Thresholding - for binarization")
print("2. Morphological Operations - for noise reduction")
print("3. Connected Components Analysis - for detection")
print("4. Contour Analysis - for alternative method")
print("5. Character Normalization - 32x32 squares")
print("\n✓ Ready for OCR/Character Recognition!")
print("="*60)